# W08 — Sealed June test of the carry-forward queue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w08_sealed_june.ipynb?flush_cache=true)

**Rules of this notebook (read before running anything):**
1. Run it **once**, top to bottom. June is queried in Section 3 only.
2. Sections 0 to 2 are written and frozen **before** June is touched. Do not edit them after you see a June number.
3. The result is saved to `work/outputs/sealed_june_metrics.json`. The notebook refuses to overwrite it.
4. Whatever the result is, it goes in the paper. A bad number is reported, not fixed.

## 0. Prediction

In plain words, first: *what do I expect June to look like, and why?*

> TODO: 2 to 3 sentences in your own words. Which number do I expect to be lower than March, and which do I expect to hold up? Why?

Then put the numbers in the next cell. The cell stops the notebook if you leave them empty.

In [2]:
#prediction
PREDICTION = {
    "precision_at_20": "between 0.20 to 0.50 with best guess around 0.35",
    "queue_top100_share": "between 0.06 to 0.11 with best guess around 0.08",
    "why": "Precision@20 depends on only 20 pages so it should swing more; both numbers should land a little below March because the method and the click rates were chosen on March data.",                     # one sentence
}
assert all(v not in (None, "") for v in PREDICTION.values()), "Fill in PREDICTION before running."
print(PREDICTION)

{'precision_at_20': 'between 0.20 to 0.50 with best guess around 0.35', 'queue_top100_share': 'between 0.06 to 0.11 with best guess around 0.08', 'why': 'Precision@20 depends on only 20 pages so it should swing more; both numbers should land a little below March because the method and the click rates were chosen on March data.'}


## 1. What is frozen

Everything here comes from W03 to W07. Nothing is re-fitted on May or June.

| Frozen choice | Value | Where it came from |
|---|---|---|
| Review date | 2026-05-31 (May complete, June not started) | mirrors W07's Feb 28 / March design |
| Tier click rates | 0.2811%, 0.2353%, 0.2226%, 0.0748% (March) | W03 |
| Queue score | May lost clicks = rate(May position tier) x May impressions - May clicks | W07 |
| Population | keyword articles, search volume >= 20, >= 50 June impressions | W07 |
| New-page cutoff | 60 days | W07 |
| June "truth" | June lost clicks, scored with the **same March rates** | **decision: see note below** |

**Decision note (yours to confirm):** scoring June with the March rates keeps the label comparable to W07 and tests whether the *queue* holds up. It does not test whether the rates themselves are stable. That is checked separately, as a diagnostic, in Section 5. If your course skill says to score differently, change this here and say why. Do it now, not after seeing June.

In [3]:
import duckdb, os, json, hashlib
import numpy as np
import pandas as pd
from google.colab import userdata

hf_token = userdata.get("HF_token")
os.environ["HF_token"] = hf_token
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

REVIEW_DATE = pd.Timestamp("2026-05-31")
NEW_DAYS = 60
RATES = np.array([0.002811, 0.002353, 0.002226, 0.000748])   # tiers 1-3, 4-10, 11-20, 21+
K = 100
SEAL_FILE = "work/outputs/sealed_june_metrics.json"
assert not os.path.exists(SEAL_FILE), "A sealed result already exists. Do not run this twice."

def tier_idx(pos):
    return np.select([pos <= 3, pos <= 10, pos <= 20], [0, 1, 2], default=3)

## 2. Build the queue from May only (no June in this section)

In [4]:
may = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-05/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    )
    SELECT content_hash_id,
           SUM(gsc_impressions) AS may_impressions,
           SUM(gsc_clicks) AS may_clicks,
           SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS may_position
    FROM daily GROUP BY content_hash_id
""").df()
may["may_lost_clicks"] = RATES[tier_idx(may["may_position"].values)] * may["may_impressions"] - may["may_clicks"]

dim = con.sql(f"""
    SELECT content_hash_id, client_hash_id, search_volume, content_created_date
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE content_type = 'keyword article' AND search_volume >= 20
""").df()

queue = dim.merge(may, on="content_hash_id", how="left").sort_values("content_hash_id").reset_index(drop=True)
queue["has_may"] = queue["may_impressions"].notna()
queue["queue_rank"] = queue["may_lost_clicks"].rank(ascending=False, method="first")

created = pd.to_datetime(queue["content_created_date"])
age_days = (REVIEW_DATE - created).dt.days
queue["unranked_group"] = np.select(
    [queue["has_may"], age_days < 0, age_days < NEW_DAYS],
    ["ranked", "not_yet_created", "new_page"], default="older_page")

# A fingerprint of the frozen top of the queue, saved with the result so the order is checkable.
top_ids = queue.nsmallest(500, "queue_rank")["content_hash_id"].tolist()
QUEUE_FINGERPRINT = hashlib.sha256("|".join(top_ids).encode()).hexdigest()
print("pages in queue pool:", len(queue), "| with May data:", int(queue["has_may"].sum()))
print("fingerprint of May top 500:", QUEUE_FINGERPRINT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages in queue pool: 114507 | with May data: 62596
fingerprint of May top 500: ede30069fbbfbde58f2a6b6599ddf2396e55d040c76bc926d12ac4304317493d


> **Optional, stronger seal:** commit and push the notebook right here, with the fingerprint printed, *before* running Section 3.

## 3. Query June once and score

In [5]:
jun = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-06/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    )
    SELECT content_hash_id,
           SUM(gsc_impressions) AS jun_impressions,
           SUM(gsc_clicks) AS jun_clicks,
           SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS jun_position
    FROM daily GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

ev = queue.merge(jun, on="content_hash_id", how="inner")     # evaluation population (same filter as W07)
ev["jun_lost_clicks"] = RATES[tier_idx(ev["jun_position"].values)] * ev["jun_impressions"] - ev["jun_clicks"]

r = ev[ev["unranked_group"] == "ranked"].copy()
pos = r["jun_lost_clicks"].clip(lower=0)

def share(ids):
    return float(pos[r["content_hash_id"].isin(ids)].sum() / pos.sum())

q_top = r.nsmallest(K, "queue_rank")["content_hash_id"]
best_top = r.nlargest(K, "jun_lost_clicks")["content_hash_id"]
imp_top = r.nlargest(K, "may_impressions")["content_hash_id"]          # no-model reference

q20 = set(r.nsmallest(20, "queue_rank")["content_hash_id"])
best20 = set(r.nlargest(20, "jun_lost_clicks")["content_hash_id"])
imp20 = set(r.nlargest(20, "may_impressions")["content_hash_id"])

metrics = {
    "review_date": str(REVIEW_DATE.date()),
    "k": K,
    "n_eval_pages": int(len(ev)),
    "n_ranked_pages": int(len(r)),
    "group_counts": ev["unranked_group"].value_counts().to_dict(),
    "queue_top_k_share": round(share(q_top), 3),
    "best_possible_top_k_share": round(share(best_top), 3),
    "random_k_share_expected": round(K / len(r), 4),
    "may_impressions_only_top_k_share": round(share(imp_top), 3),
    "queue_precision_at_20": len(q20 & best20) / 20,
    "may_impressions_only_precision_at_20": len(imp20 & best20) / 20,
    "random_precision_at_20_expected": round(20 / len(r), 4),
}
print(json.dumps(metrics, indent=2, default=str))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "review_date": "2026-05-31",
  "k": 100,
  "n_eval_pages": 38661,
  "n_ranked_pages": 36256,
  "group_counts": {
    "ranked": 36256,
    "not_yet_created": 1745,
    "new_page": 360,
    "older_page": 300
  },
  "queue_top_k_share": 0.171,
  "best_possible_top_k_share": 0.24,
  "random_k_share_expected": 0.0028,
  "may_impressions_only_top_k_share": 0.149,
  "queue_precision_at_20": 0.55,
  "may_impressions_only_precision_at_20": 0.4,
  "random_precision_at_20_expected": 0.0006
}


## 4. Save the sealed result (and the prediction next to it)

In [6]:
import sklearn
os.makedirs("work/outputs", exist_ok=True)
sealed = {
    "prediction_written_before_run": PREDICTION,
    "frozen_rates": RATES.tolist(),
    "new_page_days_cutoff": NEW_DAYS,
    "queue_fingerprint_top500_may": QUEUE_FINGERPRINT,
    "metrics": metrics,
    "versions": {"pandas": pd.__version__, "numpy": np.__version__, "duckdb": duckdb.__version__},
}
with open(SEAL_FILE, "w") as f:
    json.dump(sealed, f, indent=2, default=str)
print("Saved", SEAL_FILE, "- commit this file together with this notebook. Do not re-run.")

Saved work/outputs/sealed_june_metrics.json - commit this file together with this notebook. Do not re-run.


## 5. Diagnostics (read-only: these explain the result, they never change it)

These cells exist to tell apart *why* a result is lower. Do **not** go back and change Sections 1 to 3 after reading them.

In [7]:
# (a) Did the click rates drift? June's own impression-weighted median CTR per tier vs the frozen March rates.
def wmedian_by_tier(df):
    out = {}
    t = tier_idx(df["jun_position"].values)
    for i in range(4):
        g = df[t == i].copy()
        g["ctr"] = g["jun_clicks"] / g["jun_impressions"]
        g = g.sort_values("ctr")
        cum = g["jun_impressions"].cumsum().values
        out[i] = float(g["ctr"].values[np.searchsorted(cum, 0.5 * g["jun_impressions"].sum())])
    return out

june_rates = wmedian_by_tier(ev)
drift = pd.DataFrame({"march_rate": RATES,
                      "june_rate": [june_rates[i] for i in range(4)]})
drift["change_pct"] = (drift["june_rate"] / drift["march_rate"] - 1) * 100
drift["over_25pct"] = drift["change_pct"].abs() > 25
drift.round(6)

,march_rate,june_rate,change_pct,over_25pct
0,0.002811,0.005204,85.127099,True
1,0.002353,0.002539,7.910971,False
2,0.002226,0.002177,-2.188053,False
3,0.000748,0.000692,-7.465711,False


In [8]:
# (b) How often did a true top-100 page have a very different June vs May?
t100 = r.nlargest(K, "jun_lost_clicks")
ratio = t100["jun_impressions"] / t100["may_impressions"]
print("Median June/May impressions for the true top 100:", round(float(ratio.median()), 2))
print("Share of true top 100 with June impressions >= 3x May:", round(float((ratio >= 3).mean()), 3))

# (c) Rank agreement between the May queue score and June truth, among ranked pages.
print("Spearman (May lost clicks vs June lost clicks):",
      round(float(r[["may_lost_clicks", "jun_lost_clicks"]].corr(method="spearman").iloc[0, 1]), 3))

# (d) Monitoring triggers from W07
print("precision@20 trigger (0.30 or lower):", metrics["queue_precision_at_20"] <= 0.30)
print("older-page share of eval population:",
      round(float((ev["unranked_group"] == "older_page").mean()), 4), "(trigger at 0.02)")

Median June/May impressions for the true top 100: 0.95
Share of true top 100 with June impressions >= 3x May: 0.06
Spearman (May lost clicks vs June lost clicks): 0.489
precision@20 trigger (0.30 or lower): False
older-page share of eval population: 0.0078 (trigger at 0.02)


## 6. How to read the result (decide the reading from the diagnostics, not from hope)

| What you see | Most likely reading | What you can say |
|---|---|---|
| Queue holds up (close to March) and beats May-impressions-only | The queue is stable across one more month | "Directional support, two months" (not proof) |
| Queue drops, but a tier rate moved by more than 25% | The method (rank by gap) may be fine, but the frozen March rates went stale | Rates need monthly recomputation. This is a **limitation of the rule as built**, and it is exactly what the W07 trigger table warns about |
| Queue drops, rates are stable, many top pages have June/May >= 3x | Pages change faster than a one-month look-back can see | The method is **limited by what February/May data can show** (this matches your W06 error analysis) |
| Queue drops, rates stable, traffic changes ordinary | The March result was partly luck, or the method is weak | Report as a **weaker-than-March result**. Do not explain it away |
| Queue no better than May-impressions-only | The "gap" idea adds little beyond "busy pages" | Say so. That is a valid finding |

A bad June does not decide between "limitation" and "weak" by itself. The diagnostics above are what separate them, and even then you may have to say "I can't tell."

### Claim rewrite (fill in after the run)

- **Methods:** _TODO_
- **Observed:** _TODO (numbers from the saved JSON only)_
- **Guess:** _TODO (anything about why; label it untested)_
- **Use:** decision support only, two months of data.